# Phase 2: Song Version Embeddings + Clustering
Use filename-grouped instances and MERT embeddings to compare versions of a specific song and flag likely bad recordings.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchaudio
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModel, Wav2Vec2FeatureExtractor

from pipeline.config import load_config
from pipeline.organize import build_song_version_catalog

In [ ]:
config = load_config()
instances_path = config['ANALYZED_DIR'] / 'song_instances.csv'
summary_path = config['ANALYZED_DIR'] / 'song_versions_summary.csv'

if instances_path.exists() and summary_path.exists():
    instances_df = pd.read_csv(instances_path)
    summary_df = pd.read_csv(summary_path)
else:
    instances_df, summary_df = build_song_version_catalog(config['FILES_CSV'], exclude_hidden=True)
    config['ANALYZED_DIR'].mkdir(parents=True, exist_ok=True)
    instances_df.to_csv(instances_path, index=False)
    summary_df.to_csv(summary_path, index=False)

print(f'Loaded {len(instances_df)} instances across {len(summary_df)} song groups')
summary_df.head(20)

## Choose Song Key
Pick one canonical `song_key` from the summary table (for example: `song 05`).

In [ ]:
TARGET_SONG_KEY = 'song 05'
MIN_DURATION_SECONDS = 45
DISTANCE_THRESHOLD = 0.35
MODEL_NAME = 'm-a-p/MERT-v1-95M'
MAX_AUDIO_SECONDS = 180
CHUNK_SECONDS = 15

In [ ]:
song_df = instances_df[instances_df['song_key'] == TARGET_SONG_KEY].copy()
if song_df.empty:
    raise ValueError(f'No recordings found for song_key={TARGET_SONG_KEY!r}')

song_df['full_path'] = song_df['filepath_anonymized'].apply(lambda p: config['LOCAL_DATA_PATH'] / Path(str(p)))
song_df['exists'] = song_df['full_path'].apply(lambda p: p.exists())
song_df = song_df[song_df['exists']].copy().reset_index(drop=True)

print(f'Recordings found for {TARGET_SONG_KEY}: {len(song_df)}')
song_df[['file_stem', 'recording_date', 'version_num', 'part', 'duration_seconds', 'is_wip', 'full_path']]

## Load MERT and Extract Embeddings
On RTX 5070 Ti, `m-a-p/MERT-v1-95M` should run on CUDA. If CUDA is not available, this falls back to CPU.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
target_sr = feature_extractor.sampling_rate

print(f'Device: {device}')
print(f'Model: {MODEL_NAME}')
print(f'Target sample rate: {target_sr}')

In [ ]:
def load_audio_mono(path: Path, target_sr: int) -> torch.Tensor:
    waveform, sr = torchaudio.load(str(path))
    if waveform.ndim == 2 and waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != target_sr:
        waveform = torchaudio.functional.resample(waveform, sr, target_sr)
    return waveform.squeeze(0)


def embed_file(path: Path) -> np.ndarray:
    audio = load_audio_mono(path, target_sr)
    max_samples = int(MAX_AUDIO_SECONDS * target_sr)
    if audio.numel() > max_samples:
        audio = audio[:max_samples]

    chunk_size = int(CHUNK_SECONDS * target_sr)
    chunks = []
    start = 0
    while start < audio.numel():
        chunk = audio[start:start + chunk_size]
        if chunk.numel() == 0:
            break
        chunks.append(chunk)
        start += chunk_size

    chunk_embeddings = []
    for chunk in chunks:
        inputs = feature_extractor(
            chunk.numpy(),
            sampling_rate=target_sr,
            return_tensors='pt',
        )
        input_values = inputs.input_values.to(device)
        with torch.inference_mode():
            outputs = model(input_values=input_values)
            hidden = outputs.last_hidden_state
            emb = hidden.mean(dim=1).squeeze(0).detach().cpu().numpy()
        chunk_embeddings.append(emb)

    return np.mean(np.stack(chunk_embeddings, axis=0), axis=0)


embeddings = []
for _, row in song_df.iterrows():
    try:
        embeddings.append(embed_file(row['full_path']))
    except Exception as exc:
        print(f"Failed embedding for {row['file_stem']}: {exc}")
        embeddings.append(None)

song_df['embedding_ok'] = [e is not None for e in embeddings]
valid_df = song_df[song_df['embedding_ok']].copy().reset_index(drop=True)
valid_embeddings = [e for e in embeddings if e is not None]

print(f'Embeddings successful: {len(valid_embeddings)} / {len(song_df)}')

## Cluster Versions and Flag Bad Recording Candidates

In [ ]:
if len(valid_embeddings) < 2:
    valid_df['cluster_id'] = 0
    valid_df['outlier_score'] = 0.0
else:
    X = np.stack(valid_embeddings, axis=0)

    try:
        clusterer = AgglomerativeClustering(
            n_clusters=None,
            distance_threshold=DISTANCE_THRESHOLD,
            metric='cosine',
            linkage='average',
        )
    except TypeError:
        clusterer = AgglomerativeClustering(
            n_clusters=None,
            distance_threshold=DISTANCE_THRESHOLD,
            affinity='cosine',
            linkage='average',
        )

    labels = clusterer.fit_predict(X)
    valid_df['cluster_id'] = labels

    norms = np.linalg.norm(X, axis=1, keepdims=True)
    Xn = X / np.clip(norms, 1e-8, None)
    outlier_scores = np.zeros(len(valid_df), dtype=float)

    for cid in sorted(valid_df['cluster_id'].unique()):
        idx = np.where(valid_df['cluster_id'].values == cid)[0]
        cluster_vecs = Xn[idx]
        centroid = cluster_vecs.mean(axis=0, keepdims=True)
        centroid /= np.clip(np.linalg.norm(centroid, axis=1, keepdims=True), 1e-8, None)
        sims = (cluster_vecs @ centroid.T).squeeze(1)
        dists = 1 - sims
        if len(dists) > 1:
            z = (dists - dists.mean()) / (dists.std() + 1e-8)
            outlier_scores[idx] = z
        else:
            outlier_scores[idx] = 0.0

    valid_df['outlier_score'] = outlier_scores

valid_df['duration_seconds'] = pd.to_numeric(valid_df['duration_seconds'], errors='coerce')
valid_df['bad_candidate'] = (
    (valid_df['duration_seconds'].fillna(0) < MIN_DURATION_SECONDS)
    | (valid_df['outlier_score'] >= 1.0)
)

sim_matrix = cosine_similarity(np.stack(valid_embeddings, axis=0)) if len(valid_embeddings) > 1 else None

view_cols = [
    'file_stem', 'recording_date', 'version_num', 'part', 'duration_seconds',
    'cluster_id', 'outlier_score', 'is_wip', 'bad_candidate', 'filepath_anonymized'
]
valid_df.sort_values(['bad_candidate', 'outlier_score', 'recording_date'], ascending=[False, False, True])[view_cols]

In [ ]:
safe_song = re.sub(r'[^a-zA-Z0-9]+', '_', TARGET_SONG_KEY).strip('_').lower()
clusters_path = config['ANALYZED_DIR'] / f'song_version_clusters_{safe_song}.csv'
valid_df.to_csv(clusters_path, index=False)
print(f'Saved clustering output: {clusters_path}')

if sim_matrix is not None:
    sim_df = pd.DataFrame(sim_matrix, index=valid_df['file_stem'], columns=valid_df['file_stem'])
    sim_path = config['ANALYZED_DIR'] / f'song_similarity_{safe_song}.csv'
    sim_df.to_csv(sim_path)
    print(f'Saved similarity matrix: {sim_path}')

valid_df['bad_candidate'].value_counts()